In [22]:
import os
import pickle
from pathlib import Path

from datetime import date
from itertools import product

import numpy as np
import pandas as pd
import networkx as nx
import seaborn as sns
import matplotlib.pyplot as plt

# Setting up

In [23]:
# ==== Adaptive intimacy + RL hyperparameters ====
KAPPA = 0.08        # homophily learning rate (0–0.2)
DECAY = 0.01        # forgetting factor on intimacy
EPS   = 1e-9        # numeric stability
MIN_W = 0.002       # min intimacy weight
MAX_W = 1.0         # max intimacy weight (pre-normalization)
UPDATE_EVERY = 1    # update intimacy every k timesteps

# RL settings
USE_RL_LEADER = True          # set False to revert to threshold-based leader
RL_ACTIONS = [0, 1, 2, 3]     # 0: none, 1: weak, 2: medium, 3: strong

In [24]:
def create_intimacyMatrix(
    rng,
    population,
    intra_strength,  # within
    inter_strength,  # between
    size=None,
    min_weight=0.01,
):
    """
    Create an asymmetric, row-normalized intimacy matrix for two communities
    (different strength values) or random network (same strength values).

    NOTE: fixed assignment bug: community 0 and 1 are now properly separated.
    """
    assert population % 2 == 0, "Population (team size) must be even."

    if size is None:
        n1 = population // 2
        n2 = population - n1
    else:
        n1, n2 = size
        assert n1 + n2 == population, "size must sum to population."

    assert 0 <= intra_strength <= 1 and 0 <= inter_strength <= 1
    assert 0 < min_weight < 1

    # Randomly assign communities: 0 and 1
    assignments = np.ones(population, dtype=int)
    community0_indices = rng.choice(population, size=n1, replace=False)
    assignments[community0_indices] = 0  # 0 = community 0, 1 = community 1

    # Block upper bounds
    block_bounds = np.array([
        [intra_strength, inter_strength],
        [inter_strength, intra_strength]
    ])

    W = np.zeros((population, population), dtype=float)
    for i in range(population):
        for j in range(population):
            bound = block_bounds[assignments[i], assignments[j]]
            assert bound >= min_weight, "min_weight exceeds upper bound."
            W[i, j] = rng.uniform(min_weight, bound)

    # Normalize so each row sums to 1
    intimacyMatrix = W / W.sum(axis=1, keepdims=True)
    return intimacyMatrix, assignments

In [25]:
# Intimacy matrix for core-periphery structure
def create_corePeripheryMatrix(
    rng,
    population,
    core_proportion=0.25,    # what percentage of nodes do we want in the core
    core_to_core=0.65,      # core-core (those within the core influence); also idk how i feel with these weights; how should we even do them?
    core_to_periph=0.5,     # core -> periphery (core to periphery influence)
    periph_to_core=0.2,     # periphery -> core (periphery to core ifluence)
    periph_to_periph=0.1,    # periphery-periphery (those within the periphery influence)
    min_weight=0.01,
):
    """
    Create an asymmetric, row-normalized intimacy matrix with a core-periphery structure.
    """
    assert 0 <= core_proportion <= 1
    core_size = max(1, round(population * core_proportion))  # at least 1 core node
    #periph_size = population - core_size  # comment this out for random assignment
    
    # Assign first `core_size` as core (1), rest as periphery (0)
    #assignments = np.array([1]*core_size + [0]*periph_size)  # comment this out for random assignment
    
    # Or randomly assign core/periphery
    core_indices = rng.choice(population, size=core_size, replace=False)
    assignments = np.zeros(population, dtype=int)
    assignments[core_indices] = 1  # 1 = core, 0 = periphery

    # block bounds: [from_type, to_type]
    # 1 = core, 0 = periphery
    block_bounds = np.array([
        [periph_to_periph, periph_to_core],
        [core_to_periph, core_to_core]
    ])
    
    W = np.zeros((population, population))
    for i in range(population):
        for j in range(population):
            bound = block_bounds[assignments[i], assignments[j]]
            assert bound >= min_weight, "min_weight exceeds some upper bound; lower it."
            W[i, j] = rng.uniform(min_weight, bound)
    
    # row-normalize
    intimacyMatrix = W/W.sum(axis=1, keepdims=True)
    
    return intimacyMatrix, assignments

In [26]:
def update_intimacy_matrix(intimacy, agents):
    """
    Adaptive intimacy update:
    - Homophily: agents with similar emotions get closer (weights ↑)
    - Forgetting: small decay everywhere to avoid saturation.
    - Row-normalized; asymmetry preserved.
    """
    A = intimacy.copy()
    emos = np.array([a['emotion'] for a in agents])  # shape (N,)

    # Pairwise |q_i - q_j|
    diff = np.abs(emos[:, None] - emos[None, :])     # (N, N)

    # Homophily gain: larger when diff is small
    gain = KAPPA * (1.0 - diff)                      # in [0, KAPPA]

    # Apply decay + gain
    A = (1.0 - DECAY) * A + gain

    # No self-ties; then clamp to [MIN_W, MAX_W]
    np.fill_diagonal(A, 0.0)
    A = np.clip(A, MIN_W, MAX_W)

    # Row-normalize
    row_sums = A.sum(axis=1, keepdims=True) + EPS
    A = A / row_sums
    return A

In [27]:
# ====== INITIALIZATION ======

def initialize(rng, populationSize):
    """
    Initialize agents, leader, and intimacyMatrix.

    Uses global:
        populationSize, style, network_structure
    """
    global agents, leader, intimacyMatrix, leaderIntimacy, assignments

    agents = []
    for _ in range(populationSize):
        newAgent = {
            'emotion': -0.5 + 1 * rng.beta(2, 5),
            'delta': rng.uniform(0, 1),
            'expressiveness': rng.uniform(0, 1),
            'amplification': rng.uniform(0, 1),
            'bias': rng.uniform(0, 1)
        }
        agents.append(newAgent)

    # Randomly pick leader and remove from team
    leader = rng.choice(agents)
    agents.remove(leader)

    # Strip agent-only fields from leader
    for key in ['delta', 'expressiveness', 'amplification', 'bias']:
        if key in leader:
            del leader[key]
    leader['emotion'] = 1.0

    # Style-specific leader config (kept for backward comparability)
    if style == 'High_Fully_Constrained' or style == 'High_Initially_Constrained':
        leader.update({'emotionManagementAbility': 'High'})
        leader.update({'interventionThreshold': -0.5})
    elif style == 'Low_Fully_Constrained' or style == 'Low_Initially_Constrained':
        leader.update({'emotionManagementAbility': 'Low'})
        leader.update({'interventionThreshold': -0.7})
    # else:
    #     leader.update({'emotionManagementAbility': 'None'})
    #     leader.update({'interventionThreshold': None})  # changed from -0.3 to None because this one doesn't intervene at all (used for base comparison)

    # Assign indices to agents
    for idx, agent in enumerate(agents):
        agent['index'] = idx

    # Intimacy weights depend on network_structure
    if network_structure == "community":
        intimacyMatrix, assignments = create_intimacyMatrix(
            rng=rng, population=len(agents),
            intra_strength=0.5, inter_strength=0.2
        )
    elif network_structure == "random":
        intimacyMatrix, assignments = create_intimacyMatrix(
            rng=rng, population=len(agents),
            intra_strength=0.5, inter_strength=0.5
        )
    elif network_structure == "core_periphery":
        intimacyMatrix, assignments = create_corePeripheryMatrix(
            rng=rng, population=len(agents)
        )
    else:
        raise ValueError(f"Unknown network_structure: {network_structure}")
    
    # Create intimacy between leader and followers
    leaderIntimacy = rng.uniform(0,1, size=len(agents))  # shape (number of agents,); access via leaderIntimacy[agent['index']]

In [28]:
# ====== BOSSE EMOTION UPDATE ======

def emotional_valence_update(agentA, agentB, agentA_index, agentB_index):
    """
    Update emotional valence of two agents according to Bosse et al.
    """
    if (agentB_index, agentA_index) not in absorption_dict:
        absorption_dict[(agentB_index, agentA_index)] = 0.0

    if (agentA_index, agentB_index) not in absorption_dict:
        absorption_dict[(agentA_index, agentB_index)] = 0.0

    initial_qA = agentA['emotion']
    initial_qB = agentB['emotion']

    gamma_A = sum(sender['expressiveness'] * intimacyMatrix[sender['index'], agentA['index']] * agentA['delta'] for sender in agents if sender is not agentA)
    gamma_B = sum(sender['expressiveness'] * intimacyMatrix[sender['index'], agentB['index']] * agentB['delta'] for sender in agents if sender is not agentB)

    eta_A = agentA['amplification']
    eta_B = agentB['amplification']
    beta_A = agentA['bias']
    beta_B = agentB['bias']

    groupEmos_A = sum(cat['expressiveness'] * intimacyMatrix[cat['index'], agentA['index']] for cat in agents if cat is not agentA)
    qstar_A = sum(((sender['expressiveness'] * intimacyMatrix[sender['index'], agentA['index']]) / groupEmos_A) * sender['emotion'] for sender in agents if sender is not agentA)

    groupEmos_B = sum(cat['expressiveness'] * intimacyMatrix[cat['index'], agentB['index']] for cat in agents if cat is not agentB)
    qstar_B = sum(((sender['expressiveness'] * intimacyMatrix[sender['index'], agentB['index']]) / groupEmos_B) * sender['emotion'] for sender in agents if sender is not agentB)

    PI_A = 1 - (1 - qstar_A) * (1 - initial_qA)
    NI_A = qstar_A * initial_qA
    PI_B = 1 - (1 - qstar_B) * (1 - initial_qB)
    NI_B = qstar_B * initial_qB

    agentA['emotion'] += gamma_A * (eta_A * (beta_A * PI_A + (1 - beta_A) * NI_A) + (1 - eta_A) * qstar_A - initial_qA)
    agentA['emotion'] = np.clip(agentA['emotion'], -1, 1)
    absorption_dict[(agentA_index, agentB_index)] += abs(initial_qA - agentA['emotion'])

    agentB['emotion'] += gamma_B * (eta_B * (beta_B * PI_B + (1 - beta_B) * NI_B) + (1 - eta_B) * qstar_B - initial_qB)
    agentB['emotion'] = np.clip(agentB['emotion'], -1, 1)
    absorption_dict[(agentB_index, agentA_index)] += abs(initial_qB - agentB['emotion'])

In [29]:
def avgEmotion(agents):
    return sum(agent['emotion'] for agent in agents) / len(agents)

In [30]:
# ====== DEFINE INTERACTIONS ======

def agent_interaction(rng):
    """
    Define inter-agent interactions based on intimacy probabilities.
    """
    global agents
    buddies = []

    for i, agentA in enumerate(agents):
        for j, agentB in enumerate(agents):
            if i == j:
                continue
            interaction_prob = max(intimacyMatrix[i, j], intimacyMatrix[j, i])
            if (rng.random() < interaction_prob) and not ((i, j) in buddies or (j, i) in buddies):
                buddies.append((i, j))

    for i, j in buddies:
        agentA, agentB = agents[i], agents[j]
        emotional_valence_update(agentA, agentB, i, j)

    return buddies

In [31]:
def compute_homophily_index(agents, intimacyMatrix, tau=0.35):
    """
    Scalar homophily index at current timestep.

    Intuition:
      - "Similar" pairs: |emotion_i - emotion_j| <= tau
      - "Dissimilar" pairs: |emotion_i - emotion_j|  > tau
      - Index = mean(intimacy of similar pairs) - mean(intimacy of dissimilar pairs)

    Higher index => more weight on similar-emotion ties (stronger homophily).
    """
    emos = np.array([a['emotion'] for a in agents])
    W = intimacyMatrix.copy()

    # Exclude self-ties from metric
    np.fill_diagonal(W, 0.0)

    N = len(emos)
    similar_weights = []
    dissimilar_weights = []

    for i in range(N):
        for j in range(N):
            if i == j:
                continue
            diff = abs(emos[i] - emos[j])
            w = W[i, j]
            if diff <= tau:
                similar_weights.append(w)
            else:
                dissimilar_weights.append(w)

    if len(similar_weights) == 0 or len(dissimilar_weights) == 0:
        # If everything is similar or everything is dissimilar, just return 0
        return 0.0

    return float(np.mean(similar_weights) - np.mean(dissimilar_weights))

# RL Set-up

In [32]:
# ====== RL STATE, ACTIONS, REWARD ======

def compute_state(agents, intimacyMatrix):
    """
    RL state: [mean_emotion, var_emotion, homophily_index].
    The third component is the scalar homophily index, not plain avg intimacy.
    """
    emos = np.array([a['emotion'] for a in agents])
    mean_emotion = emos.mean()
    var_emotion = emos.var()

    # use our scalar homophily metric
    homo = compute_homophily_index(agents, intimacyMatrix)

    return np.array([mean_emotion, var_emotion, homo], dtype=float)

def apply_leader_action(action, agents, leader):
    """
    Map discrete RL action to an intervention strength and apply it.
    action in {0,1,2,3} = {none, weak, medium, strong}
    """
    if action == 0:
        return

    if action == 1:
        dampening = 0.02
    elif action == 2:
        dampening = 0.05
    elif action == 3:
        dampening = 0.08
    else:
        raise ValueError(f"Unknown action: {action}")

    for agent in agents:
        agent['emotion'] += dampening * (leader['emotion'] - agent['emotion']) * agent['delta'] * leaderIntimacy[agent['index']]
        agent['emotion'] = np.clip(agent['emotion'], -1, 1)

def compute_quality(agents, w1=1.0, w2=0.5):
    """
    Overall 'quality' = high if agents are happy & similar.
    q = w1 * mean_emotion - w2 * var_emotion
    """
    emos = np.array([a['emotion'] for a in agents])
    mean_emotion = emos.mean()
    var_emotion = emos.var()
    return w1 * mean_emotion - w2 * var_emotion

In [33]:
# ====== Q‑LEARNING LEADER POLICY ======
class QLearningLeaderPolicy:
    """
    Tabular Q-learning over a discretised state:
      state = [mean_emotion, var_emotion, avg_intimacy]

    Q[bin_mean, bin_var, bin_homo, action] stores the value of
    taking 'action' in that coarse state.
    """

    def __init__(
        self,
        actions,
        n_bins_mean=5,
        n_bins_var=4,
        n_bins_homo=4,
        alpha=0.1,
        gamma=0.95,
        epsilon_start=0.3,
        epsilon_end=0.05,
        epsilon_decay_steps=2000,
        ):
        self.actions = list(actions)
        self.nA = len(self.actions)

        self.n_bins_mean = n_bins_mean
        self.n_bins_var = n_bins_var
        self.n_bins_homo = n_bins_homo

        # Learning hyper‑params
        self.alpha = alpha
        self.gamma = gamma

        # epsilon‑greedy schedule
        self.epsilon = epsilon_start
        self.epsilon_start = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay_steps = max(1, epsilon_decay_steps)
        self.step_count = 0

        # Q‑table
        self.Q = np.zeros(
            (n_bins_mean, n_bins_var, n_bins_homo, self.nA),
            dtype=float
            )

    # --- Helpers to discretise continuous state ---

    def _bin_value(self, value, vmin, vmax, n_bins):
        # Clamp into [vmin, vmax] then map to {0,...,n_bins-1}
        value_clipped = max(vmin, min(vmax, value))
        frac = (value_clipped - vmin) / (vmax - vmin + 1e-9)
        idx = int(frac * n_bins)
        if idx == n_bins:  # edge case at vmax
            idx -= 1
        return idx

    def _state_indices(self, state):
        """
        state = np.array([mean_emotion, var_emotion, avg_intimacy])
        We assume rough ranges:
          mean_emotion in [-1, 1]
          var_emotion  in [0, 1]
          avg_intimacy in [0, 0.05]  (adjust if your homophily range changes)
        """
        m, v, h = state
        i_m = self._bin_value(m, -1.0, 1.0, self.n_bins_mean)
        i_v = self._bin_value(v,  0.0, 1.0, self.n_bins_var)
        i_h = self._bin_value(h,  0.0, 0.05, self.n_bins_homo)
        return (i_m, i_v, i_h)

    # --- Policy interface ---

    def choose_action(self, state, rng):
        """
        epsilon-greedy action selection.
        """
        idx = self._state_indices(state)

        # Decay epsilon over time
        self.step_count += 1
        frac = min(1.0, self.step_count / self.epsilon_decay_steps)
        self.epsilon = self.epsilon_start + frac * (self.epsilon_end - self.epsilon_start)

        if rng.random() < self.epsilon:
            return rng.choice(self.actions)

        q_vals = self.Q[idx]   # shape (nA,)
        best_a_idx = int(np.argmax(q_vals))
        return self.actions[best_a_idx]

    def update(self, state, action, reward, next_state, done=False):
        """
        Standard Q‑learning update:
            Q(s,a) ← Q(s,a) + α [ r + γ max_a' Q(s',a') − Q(s,a) ]
        If done=True we omit the future value term.
        """
        s_idx  = self._state_indices(state)
        sp_idx = self._state_indices(next_state)

        a_idx = self.actions.index(action)

        q_sa = self.Q[s_idx + (a_idx,)]

        if done:
            target = reward
        else:
            max_q_sp = np.max(self.Q[sp_idx])
            target = reward + self.gamma * max_q_sp

        self.Q[s_idx + (a_idx,)] = q_sa + self.alpha * (target - q_sa)

# Visualization

In [34]:
# def sentiment_evolution_graph(intervention_timesteps, run_id, intervention_action):
#     """
#     Plot time series of individual emotions + average emotion.
#     """
#     emotion_array = np.array(emotion_history)
#     avg_array = np.array(avg_emotion_history)
#     number_of_agents = emotion_array.shape[1]
#     flat_interventions = sorted(set(intervention_timesteps))

#     fig, ax = plt.subplots(figsize=(12, 6))
#     for i in range(number_of_agents):
#         ax.plot(emotion_array[:, i], alpha=0.4, color='gray')

#     for t in flat_interventions:
#         ax.axvline(x=t, color='tab:blue', linestyle='--', alpha=0.5,
#             label='Leader Intervention' if t == flat_interventions[0] else ""
#         )

#     ax.plot(avg_array, color='red', linewidth=2, label='Average Emotion')
#     ax.set_xlabel('Time Step')
#     ax.set_ylabel('Emotion Value')
#     ax.set_title(f'Sentiment Dynamics Over Time\n{style}, Simulation {run_id}')
#     ax.grid(True)
#     ax.set_ylim(-1, 1)
#     ax.legend()
#     plt.close()
#     return fig

In [35]:
def sentiment_evolution_graph(intervention_log, run_id):
    """
    Plot time series of individual emotions + average emotion
    with colored intervention lines.
    
    intervention_log: dict {timestep: action}
                      action in {1=weak, 2=medium, 3=strong}
    """

    emotion_array = np.array(emotion_history)
    avg_array = np.array(avg_emotion_history)
    number_of_agents = emotion_array.shape[1]

    # This replaces flat_interventions in a SAFE way
    flat_interventions = sorted(intervention_log.keys())

    color_map = {
        1: "tab:blue",    # weak
        2: "tab:orange",  # medium
        3: "tab:purple"      # strong
    }

    label_map = {
        1: "Weak Intervention",
        2: "Medium Intervention",
        3: "Strong Intervention"
    }

    fig, ax = plt.subplots(figsize=(12, 6))

    # Plot individual emotions
    for i in range(number_of_agents):
        ax.plot(emotion_array[:, i], alpha=0.4, color='gray')

    # Plot colored intervention lines
    used_labels = set()
    for t in flat_interventions:
        action = intervention_log[t]
        color = color_map.get(action, "black")
        label = label_map.get(action, "Intervention")

        show_label = label if label not in used_labels else ""
        used_labels.add(label)

        ax.axvline(
            x=t,
            color=color,
            linestyle='--',
            alpha=0.6,
            label=show_label
        )

    # Plot average emotion
    ax.plot(avg_array, color='red', linewidth=2, label='Average Emotion')

    ax.set_xlabel('Time Step')
    ax.set_ylabel('Emotion Value')
    ax.set_title(f'Sentiment Dynamics Over Time\n{style}, Simulation {run_id}')
    ax.grid(True)
    ax.set_ylim(-1, 1)
    ax.legend()

    plt.close()
    return fig

In [36]:
def social_network_graph(buddies_per_timestep, absorption_dict):
    from collections import Counter
    flat_buddies = [item for sublist in buddies_per_timestep for item in sublist]

    G = nx.Graph()
    G.add_nodes_from([agent['index'] for agent in agents])
    edge_weights_dict = Counter(flat_buddies)
    G.add_weighted_edges_from([(i, j, weight) for (i, j), weight in edge_weights_dict.items()])

    DG = nx.DiGraph()
    DG.add_nodes_from(G)
    DG.add_weighted_edges_from([(i, j, weight) for (i, j), weight in absorption_dict.items()])

    return G, DG

# Run the simulation

In [37]:
leader_behaviors = {
    "No_Intervention": {
        "uses_rl": False,  # USE_RL_LEADER
        "threshold_mode": "never",   # never constrained
    },
    "High_Fully_Constrained": {
        "uses_rl": True,
        "threshold_mode": "always",  # always constrained
    },
    "Low_Fully_Constrained": {
        "uses_rl": True,
        "threshold_mode": "always",
    },
    "High_Initially_Constrained": {
        "uses_rl": True,
        "threshold_mode": "initial", # constrained only until first intervention
    },
    "Low_Initially_Constrained": {
        "uses_rl": True,
        "threshold_mode": "initial",
    },
    "Free": {
        "uses_rl": True,
        "threshold_mode": "never",   # never constrained
    }
}

In [38]:
def run_simulation(seed, run_id, policy=None, behavior=None, populationSize=None):
    """
    One simulation run with:
    - Emotion contagion
    - Adaptive intimacy (homophily + forgetting)
    - Optional RL leader (Q-learning)
    """
    global time, emotion_history, avg_emotion_history, avg_emotional_valence, absorption_dict, intimacyMatrix

    rng = np.random.default_rng(seed)
    initialize(rng, populationSize)

    characteristics = ['emotion', 'delta', 'expressiveness', 'amplification', 'bias']
    initial_conditions = pd.DataFrame([{k: agent[k] for k in characteristics} for agent in agents])

    emotion_history = []
    emotion_history.append([agent['emotion'] for agent in agents])
    avg_emotion_history = []
    buddies_per_timestep = []
    interactions_per_timestep = []
    #intervention_timesteps = []
    intervention_log = {}
    absorption_dict = {}
    intimacy_history = []

    # Keep homophily at each step
    homophily_history = []
    homophily_history.append(compute_homophily_index(agents, intimacyMatrix))  # homophily at t = 0 (before any interaction)

    # RL logs
    actions_history = []
    rewards_history = []
    quality_history = []

    # Initial state/quality for RL
    state_t = compute_state(agents, intimacyMatrix)
    prev_quality = compute_quality(agents)
    quality_history.append(prev_quality)

    threshold_mode = behavior["threshold_mode"]
    leader_intervened = False

    time = 0
    while time < max_iterations:
        done = (time == max_iterations - 1)

        # 1) RL: choose action
        if policy is None:  # No Intervention leader
            action_t = 0
        else:
            # ----- free leader -----
            if threshold_mode == "never":
                action_t = policy.choose_action(state_t, rng)

            # ----- Fully constrained -----
            elif threshold_mode == "always":
                if avgEmotion(agents) <= leader["interventionThreshold"]:
                    action_t = policy.choose_action(state_t, rng)
                else:
                    action_t = 0

            # ----- Initially constrained -----
            elif threshold_mode == "initial":
                if not leader_intervened:
                    # Constrained BEFORE first intervention
                    if avgEmotion(agents) <= leader["interventionThreshold"]:
                        action_t = policy.choose_action(state_t, rng)
                    else:
                        action_t = 0
                else:
                    # Unconstrained AFTER first intervention
                    action_t = policy.choose_action(state_t, rng)

        actions_history.append(action_t)

        # 2) Agent interactions
        buddies = agent_interaction(rng)
        buddies_per_timestep.append(buddies)
        interactions_per_timestep.append(len(buddies))

        avg_emotional_valence = avgEmotion(agents)
        avg_emotion_history.append(avg_emotional_valence)

        # # 3) Leader intervention according to action
        # if USE_RL_LEADER and policy is not None:
        #     apply_leader_action(action_t, agents, leader)
        #     if action_t != 0:
        #         intervention_timesteps.append(time)
        #         leader_intervened = True

        # 3) Leader intervention according to action
        if USE_RL_LEADER and policy is not None:
            apply_leader_action(action_t, agents, leader)

            if action_t != 0:
                intervention_log[time] = action_t   # store action taken at this timestep
                leader_intervened = True


        # 4) Adaptive intimacy update
        if (time % UPDATE_EVERY) == 0:
            intimacyMatrix = update_intimacy_matrix(intimacyMatrix, agents)
            intimacy_history.append(intimacyMatrix)

        # 5) Log homophily after emotions + intimacy were updated
        h_t = compute_homophily_index(agents, intimacyMatrix)
        homophily_history.append(h_t)

        # 6) RL update: reward = Δquality
        if USE_RL_LEADER and policy is not None:
            new_quality = compute_quality(agents)
            reward_t = new_quality - prev_quality
            prev_quality = new_quality

            state_tp1 = compute_state(agents, intimacyMatrix)
            policy.update(state_t, action_t, reward_t, state_tp1, done=done)

            rewards_history.append(reward_t)
            quality_history.append(new_quality)

            state_t = state_tp1

        # 7) log emotions for plotting
        emotion_history.append([agent['emotion'] for agent in agents])
        time += 1

    # Build networks and sentiment figure (you might not use them for analysis now)
    networkG, networkDG = social_network_graph(buddies_per_timestep, absorption_dict)
    #sentiment_fig = sentiment_evolution_graph(intervention_timensteps, run_id)
    sentiment_fig = sentiment_evolution_graph(intervention_log, run_id)


    leader_data = {
        'emotionManagementAbility': leader.get('emotionManagementAbility'),
        'interventionThreshold': leader.get('interventionThreshold'),
        'final_avg_emotion': avg_emotion_history[-1] if avg_emotion_history else None,
        'final_agent_emotions': [agent['emotion'] for agent in agents],
    }

    results = {
        'leader_data': leader_data,
        'emotion_history': emotion_history,
        'avg_emotion_history': avg_emotion_history,
        #'intervention_timesteps': intervention_timesteps,
        'intervention_log': intervention_log,
        'final_avg_emotion': avg_emotional_valence,
        'network_graph': networkG,
        'network_digraph': networkDG,
        'absorption_dict': absorption_dict,
        'initial_conditions': initial_conditions,
        'sentiment_graph': sentiment_fig,
        'intimacy_matrix': intimacy_history,
        'rl_actions': actions_history,
        'rl_rewards': rewards_history,
        'rl_quality': quality_history,
        'homophily_index': homophily_history,
    }
    return results

In [39]:
from dataclasses import dataclass
from typing import List, Dict, Any

@dataclass
class AllSimulationResults:
    # Core
    initial_conditions: List[pd.DataFrame]      # [run]
    emotion_history: List[List[List[float]]]   # [run][t][agent]
    avg_emotions: List[List[float]]            # [run][t]
    interventions: List[List[int]]             # [run][t_intervention]

    # RL logs
    rl_actions: List[List[int]]                # [run][t]
    rl_rewards: List[List[float]]              # [run][t]
    rl_quality: List[List[float]]              # [run][t]

    # Homophily/Intimacy
    homophily_index: List[List[float]]        # [run][t]
    intimacy_history: List[List[np.ndarray]]  # [run][t] intimacyMatrix

    # Absorption
    absorption_history: List[List[Dict[tuple, float]]]  # [run][t][(i,j)] absorption from i to j

    # Visuals
    sentiment_graphs: List[Any]               # [run] matplotlib figures

    # Metadata
    meta: Dict[str, Any]

In [40]:
def run_multiple_simulations(runs, style, policy=None, populationSize=None):
    """
    Run multiple simulations and keep only the variables
    needed for:
      - RL learning / leader behavior
      - Homophily in adaptive network
    """
    all_initial_conditions = []  # i mean, there's just the one...
    all_emotion_histories = []
    all_avg_emotion = []
    all_interventions = []

    all_rl_actions = []
    all_rl_rewards = []
    all_rl_quality = []

    all_homophily = []
    all_intimacy_matrices = []

    all_absorption_dicts = []

    all_sentiment_graphs = []

    for run in range(runs):
        global agents, leader, emotion_history, avg_emotion_history

        behavior = leader_behaviors[style]
        if behavior["uses_rl"]:
            policy = QLearningLeaderPolicy(
                RL_ACTIONS,
                n_bins_mean=5,
                n_bins_var=4,
                n_bins_homo=4,
                alpha=0.1,
                gamma=0.95,
                epsilon_start=0.3,
                epsilon_end=0.05,
                epsilon_decay_steps=2000
                )
        else:
            policy = None

        results = run_simulation(seed=run, run_id=run, policy=policy, behavior=behavior, populationSize=populationSize)

        all_initial_conditions.append(results['initial_conditions'])
        
        all_emotion_histories.append(results['emotion_history'])
        all_avg_emotion.append(results['avg_emotion_history'])
        #all_interventions.append(results['intervention_timesteps'])
        all_interventions.append(results['intervention_log'])

        all_rl_actions.append(results['rl_actions'])
        all_rl_rewards.append(results['rl_rewards'])
        all_rl_quality.append(results['rl_quality'])

        all_homophily.append(results['homophily_index'])
        all_intimacy_matrices.append(results['intimacy_matrix'])

        all_absorption_dicts.append(results['absorption_dict'])

        all_sentiment_graphs.append(results['sentiment_graph'])

    # Store meta, including final learned Q-values if using RL
    if USE_RL_LEADER and policy is not None and hasattr(policy, "Q"):
        Q_estimates = policy.Q.copy()   # just store the array
    else:
        Q_estimates = None

    meta = {
        "runs": runs,
        "populationSize": populationSize,
        "max_iterations": max_iterations,
        "network_structure": network_structure,
        "style": style,
        "USE_RL_LEADER": USE_RL_LEADER,
        "KAPPA": KAPPA,
        "DECAY": DECAY,
        "UPDATE_EVERY": UPDATE_EVERY,
        "RL_ACTIONS": RL_ACTIONS,
        "Q_estimates": Q_estimates,
        }

    return AllSimulationResults(
        initial_conditions=all_initial_conditions,
        emotion_history=all_emotion_histories,
        avg_emotions=all_avg_emotion,
        interventions=all_interventions,
        rl_actions=all_rl_actions,
        rl_rewards=all_rl_rewards,
        rl_quality=all_rl_quality,
        homophily_index=all_homophily,
        intimacy_history=all_intimacy_matrices,
        absorption_history=all_absorption_dicts,
        sentiment_graphs=all_sentiment_graphs,
        meta=meta
        )

In [ ]:
# ====== TOP-LEVEL LOOP ======

team_sizes = [11, 51, 101]  # add other sizes later
network_structures = ["core_periphery", "random", "community"]
styles = ["No_Intervention", "High_Fully_Constrained", "Low_Fully_Constrained", "High_Initially_Constrained", "Low_Initially_Constrained", "Free"]  # ["High_Aperture", "Low_Aperture", "No_Intervention"]
runs = 20
#populationSize = 15  # 1 leader + 2n agents (even team size for simplicity)
max_iterations = 400
todaysDate = date.today().strftime("%m_%d_%Y")
parentfolder = Path(r"R:\sescott1\Masters\523\Simulation\RL")  # change to a valid path on your system
#parentfolder = Path(r"C:\Users\sarah\OneDrive\Documents\Masters\523\Emotion Contagion\Simulation Runs\RL")

for populationSize in team_sizes:
    for network_structure, style in product(network_structures, styles):
        print(f"Running simulations for {populationSize}, {network_structure}, {style}...")

        all_results = run_multiple_simulations(runs=runs, style=style, policy=None, populationSize=populationSize)

        # Simple numeric summary for intuition
        final_avg_list = [run[-1] for run in all_results.avg_emotions]
        num_interventions = [len(iv) for iv in all_results.interventions]
        mean_quality_per_run = [np.mean(q) if len(q) > 0 else np.nan for q in all_results.rl_quality]

        summary_df = pd.DataFrame({
            "Run": np.arange(runs),
            "Final_Avg_Emotion": final_avg_list,
            "Num_Interventions": num_interventions,
            "Mean_Quality": mean_quality_per_run
            })

        # Save results
        datefolder = Path(parentfolder) / todaysDate
        teamsizefolder = datefolder / f"team_size_{populationSize}"
        networkfolder = teamsizefolder / network_structure
        stylefolder = networkfolder / style
        stylefolder.mkdir(parents=True, exist_ok=True)
        results_folder = stylefolder

        all_results_path = results_folder / "all_results.pkl"
        with open(all_results_path, "wb") as f:
            pickle.dump(all_results, f)

        print(
            f"✅ Completed {populationSize}, {network_structure}, {style}. Results saved to {results_folder}",
            f"Running simulations: {populationSize}, {network_structure}, {style}"
        )

Running simulations for 15, core_periphery, No_Intervention...
✅ Completed 15, core_periphery, No_Intervention. Results saved to R:\sescott1\Masters\523\Simulation\RL\01_11_2026\team_size_15\core_periphery\No_Intervention Running simulations: 15, core_periphery, No_Intervention
Running simulations for 15, core_periphery, High_Fully_Constrained...
✅ Completed 15, core_periphery, High_Fully_Constrained. Results saved to R:\sescott1\Masters\523\Simulation\RL\01_11_2026\team_size_15\core_periphery\High_Fully_Constrained Running simulations: 15, core_periphery, High_Fully_Constrained
Running simulations for 15, core_periphery, Low_Fully_Constrained...
✅ Completed 15, core_periphery, Low_Fully_Constrained. Results saved to R:\sescott1\Masters\523\Simulation\RL\01_11_2026\team_size_15\core_periphery\Low_Fully_Constrained Running simulations: 15, core_periphery, Low_Fully_Constrained
Running simulations for 15, core_periphery, High_Initially_Constrained...
✅ Completed 15, core_periphery, High_